# RLHF Alignment

> Pretrained models predict the next token, but they don't distinguish what *should* be said from what *could* be said. Ask "how to make a bomb" and it will happily answer -- because the training data contains those continuation patterns. How do we teach a model "what to say and what not to say"?
>
> This section walks through the full RLHF pipeline: SFT as a foundation, Reward Model to turn preferences into scores, PPO to optimize the policy, and DPO to skip the scorer and learn preferences directly. Each step is explained with hand-calculated examples.

The training objective of a language model is "predict the next token," but what we want is "give helpful, honest, and harmless answers." The gap between these two objectives is exactly what Alignment aims to solve. The classic RLHF pipeline has three stages: first, use high-quality conversation data for SFT as a foundation; then train a Reward Model to simulate human preferences; finally, use PPO reinforcement learning to optimize the model toward higher rewards.

DPO is a simplification proposed in 2023 -- through algebraic transformation, the explicit Reward Model is eliminated, and preference data is directly converted into a classification loss.

SFT, Reward Model, PPO, DPO -- each stage has a different loss function and training objective. This section starts from the 3H principles, builds a foundation with SFT, then progressively adds more complex training objectives.


## 1. Why Alignment Is Needed

The goal of a pretrained model is "text continuation" -- given the previous N tokens, predict the N+1th token. It has learned statistical patterns of language from massive internet text, but it doesn't understand what makes an answer "helpful."

Consider this example:

```
User: I've been feeling down lately. I don't feel like life is worth living.

Unaligned Base Model (continuation mode):
  "She stood on the rooftop, the wind blowing through her hair, thinking of
   the person who betrayed her... The crowd below gathered to watch, someone
   pulled out a phone to record..."
   ^ The model is imitating novels it saw online! Statistically this is correct --
     "feeling down" is indeed often followed by this kind of literary description.
     But this response is completely inappropriate.

Aligned Model (Helpful + Harmless):
  "I'm sorry you're going through this. Please remember: you are not alone,
   and there are many people willing to help you. You can call a crisis
   hotline. Would you like to talk about what happened?"
   ^ The model recognizes this as a cry for help and provides a helpful response.
```

This contrast reveals the fundamental problem of alignment: pretraining teaches the model "what usually follows this text," but alignment must teach it "what *should* be responded in this situation." The former is about statistical patterns; the latter is about value judgment.

The 3H principles of alignment:
- **Helpful**: Answer questions, provide useful information, stay on topic
- **Honest**: Admit when it doesn't know, don't fabricate plausible-sounding falsehoods
- **Harmless**: Refuse harmful requests, don't enable dangerous behavior

These three principles sometimes conflict. For example, if a user asks "why didn't the bomb I made go off," an honest answer would provide dangerous information -- helpful to this user but harmful to society. Alignment needs to balance the three -- usually prioritizing harmlessness.


## 2. The Full Alignment Pipeline

The entire chain is divided into several stages:

```
  Base Model (pretrained, only does continuation)
       |
       v  Stage 1: SFT
  +---------------------------+
  | Supervised learning with  |
  | high-quality conversation |
  | data. Teaches dialogue    |
  | format and basic instructions |
  +-------------+--------------+
                v
          SFT Model (can converse, but can't distinguish good from bad)
                |
          +-----+-----+
          v             v
     Stage 2a: RLHF    Stage 2b: DPO
     (classic route)    (simplified route)
          |                  |
     1. Collect preference  1. Collect preference
        data                  data
     2. Train Reward Model  2. Directly optimize
     3. PPO reinforcement      preferences
        learning              (skip RM + PPO)
          |                  |
          +------+-----------+
                 v
           Aligned Model
```

Let's work through each stage with hand calculations.


## 3. Stage 1: SFT

#### 3.1 What SFT Does

Supervised training with high-quality conversation data. The training data looks like this:

```
<|User|> What is the capital of France?
<|Assistant|> The capital of France is Paris.
<|User|> Why Paris?
<|Assistant|> Because Paris is the largest city in France and its political
center, serving as the administrative capital since the Middle Ages.
```

SFT is essentially the same as pretraining -- cross-entropy loss, predicting the next token. The only difference is that the data changes from "internet text" to "conversations."

After SFT, the model has learned:
1. Dialogue format (question-answer pairs)
2. Basic instruction following
3. Surface features of "good answers"

#### 3.2 But SFT Has a Fundamental Limitation

For questions with a single correct answer like "What is the capital of France?", SFT is sufficient.

But for open-ended questions, like "Help me write a resignation letter" -- there is no unique "standard answer." Two resignation letters can both be acceptable, but which one is better? SFT's training signal cannot distinguish between them.

This is where the next stage comes in: **having human annotators tell us which answer is better.**


#### 3.3 Loss Masking: Only Computing Loss on the Response During SFT

As mentioned earlier, SFT training data is a complete conversation, typically in this format:

```
[system prompt] [user message] [assistant response]
```

The model's task is "predict the next token." If the entire data sequence is used for training, the model will simultaneously learn to predict the system prompt, user message, and assistant response. But we only want it to learn to **generate the response part** -- after all, users don't expect the model to predict their own questions.

The solution is straightforward: mark the tokens in the prompt portion with a special label so that the loss function ignores them. In PyTorch, this special label is `-100` (the default `ignore_index` for `CrossEntropyLoss`). This is called **Loss Masking**.

The specific approach:
- Set the label for prompt tokens (system + user) to `-100`, so loss is not computed
- Set the label for response tokens (assistant) normally to the next token's ID

This way, cross-entropy loss only accumulates on response tokens, and the model only learns "how to respond" rather than being trained to reproduce the prompt.


In [ ]:
import numpy as np
# === Loss Masking Demo ===
print("=== Loss Masking: Only Computing Loss on the Response ===")
print()

# Simulate an SFT training example
# Assume the tokenizer encoded the conversation into 9 tokens
# The first 5 are prompt (system + user), the last 4 are assistant's response

tokens = [1, 5, 3, 8, 2, 7, 9, 4, 6]  # Complete token sequence
prompt_len = 5  # First 5 tokens belong to the prompt

# Standard labels: label at each position = next token (shifted right by one)
labels_shifted = tokens[1:] + [-1]  # [5, 3, 8, 2, 7, 9, 4, 6, -1]

# Loss Masking: replace labels for prompt portion with -100
labels_masked = labels_shifted.copy()
for i in range(prompt_len):
    labels_masked[i] = -100  # Ignore prompt portion

print("tokens:      ", tokens)
print("labels(orig):", labels_shifted)
print("labels(mask):", labels_masked)
print()

# Use simple simulated data to demonstrate loss computation
# Assume the model outputs a probability distribution at each position;
# we simplify using "whether the prediction is correct"
# log_probs[i] = log probability of the model predicting tokens[i+1] at position i
np.random.seed(42)
log_probs = np.random.uniform(-3.0, -0.5, size=len(tokens))
log_probs = np.round(log_probs, 2)

print("--- Without Loss Masking ---")
total_loss = 0
count = 0
for i in range(len(tokens) - 1):  # Last position has no label
    label = labels_shifted[i]
    loss_i = -log_probs[i]  # Cross-entropy = -log(p)
    total_loss += loss_i
    count += 1
    print(f"  Position {i}: token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_no_mask = total_loss / count
print(f"  Average loss = {avg_no_mask:.4f} (all {count} positions participate)")
print()

print("--- With Loss Masking ---")
total_loss = 0
count = 0
for i in range(len(tokens) - 1):
    label = labels_masked[i]
    if label == -100:
        print(f"  Position {i}: token={tokens[i]}, label=-100 -> skipped (prompt)")
        continue
    loss_i = -log_probs[i]
    total_loss += loss_i
    count += 1
    print(f"  Position {i}: token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_masked = total_loss / count
print(f"  Average loss = {avg_masked:.4f} (only {count} response positions)")
print()
print("Key observation: Loss Masking prevents prompt tokens from participating in gradient updates.")
print("The model only learns from the response portion, resulting in a cleaner SFT training signal.")


## 4. Stage 2: Reward Model

#### 4.1 What Preference Data Looks Like

For a given prompt, annotators see two answers and choose the better one:

```
Prompt: Help me write a resignation letter

Answer A (chosen):
  "Dear Manager: Due to personal reasons, I have decided to resign from
   my current position. Thank you for the training and opportunities the
   company has given me over the past two years. I will ensure all handover
   work is completed before my departure. Wishing the company continued success!"

Answer B (rejected):
  "Resigning? Just write 'I quit' and be done with it.
   The company won't care whether you leave anyway."
```

The annotator chose A. This is one preference data point: `(prompt, chosen, rejected)`.

#### 4.2 What the Reward Model Does

Train a model (Reward Model, RM) that takes `(prompt, answer)` as input and outputs a score r.

The training objective is simple: **make the RM give a higher score to chosen than to rejected.**

This is essentially training an "automatic scorer" to replace human annotators.

#### 4.3 Loss Function Hand Calculation -- This Is the Bradley-Terry Model


In [ ]:
import math
# === Reward Model Loss Hand Calculation ===
print("=== Reward Model Loss Function ===")
print()
print("Formula: L = -log( sigma(r_chosen - r_rejected) )")
print("         where sigma(x) = 1/(1+e^(-x)) is the sigmoid function")
print()

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def reward_loss(r_chosen, r_rejected):
    diff = r_chosen - r_rejected
    prob = sigmoid(diff)
    loss = -math.log(max(prob, 1e-10))
    return loss, diff, prob

# Four scenarios
cases = [
    ("Good: chosen >> rejected", 8.0, 2.0),
    ("OK: chosen slightly better", 6.0, 5.0),
    ("Bad: chosen < rejected", 3.0, 7.0),
    ("Disaster: chosen << rejected", 1.0, 9.0),
]

print(f"{'Scenario':<30s} {'r_c':>6s} {'r_r':>6s} {'diff':>8s} {'sigma(diff)':>12s} {'loss':>10s}")
print("-" * 70)

for desc, r_c, r_r in cases:
    loss, diff, prob = reward_loss(r_c, r_r)
    print(f"{desc:<30s} {r_c:>6.1f} {r_r:>6.1f} {diff:>8.1f} {prob:>12.4f} {loss:>10.4f}")

print()
print("Interpretation:")
print("  * sigma(diff) is 'the probability that chosen should win'")
print("  * When r_chosen >> r_rejected: sigma~1, loss~0 -> RM got it right, small penalty")
print("  * When r_chosen << r_rejected: sigma~0, loss is large -> RM got it wrong, heavy penalty")
print()
print("This essentially turns 'preference comparison' into a binary classification problem:")
print("  label=1 (chosen is better), prediction=sigma(r_c - r_r), loss=cross_entropy")


## 5. Stage 3: PPO

With a Reward Model, we can use reinforcement learning to optimize the LLM.

#### 5.1 PPO Training Loop

```
Repeat the following steps:
  1. Take a batch of prompts (e.g., 256)
  2. LLM generates a response for each prompt
  3. RM scores each (prompt, answer) pair -> get reward
  4. Use PPO to update the LLM -- increase high-scoring tokens, decrease low-scoring ones
  5. But don't stray too far -- add KL penalty (prevent the model from forgetting what SFT taught)
```

#### 5.2 PPO's Core Mechanism: Clipping

The most critical design in PPO is a mechanism called **clip**. Here's the idea in one page:

```
ratio = new model's probability of choosing this token / old model's probability of choosing this token

If ratio = 1.5 -> old model chose this token with probability 2%, new model 3%
If ratio = 0.8 -> old model chose this token with probability 5%, new model 4%

Clip's role: constrain ratio to [1-epsilon, 1+epsilon] (typically epsilon=0.2)
  -> Prevent overly aggressive single-step updates
  -> Like a steering wheel limiter -- prevents you from turning too hard and rolling over
```


In [ ]:
import numpy as np

# === PPO Clip Mechanism Hand Calculation ===
print("=== PPO Clip Mechanism ===")
print()

def ppo_loss(ratio, advantage, epsilon=0.2):
    """
    Compute PPO surrogate loss
    ratio = pi_new / pi_old (probability ratio between new and old policies)
    advantage = RM_score - baseline (how much better this response is than average)
    """
    # Loss without clipping
    unclipped = ratio * advantage
    # Loss with clipping (constrain ratio to [0.8, 1.2])
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    # PPO takes the more conservative option (min for advantage>0, max for advantage<0)
    # This is -min(unclipped, clipped) as the surrogate
    return -min(unclipped, clipped)


ratios = [0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0]

print("When advantage > 0 (this response is good, want to increase its probability):")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {'Note'}")
print("-" * 60)
for r in ratios:
    unclipped = r * 1.0
    clipped = min(r, 1.2) * 1.0
    loss = -min(unclipped, clipped)
    note = ""
    if r > 1.2:
        note = "<- clip! Don't let it grow further"
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print("When advantage < 0 (this response is bad, want to decrease its probability):")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {'Note'}")
print("-" * 60)
for r in ratios:
    unclipped = r * (-1.0)
    clipped = max(r, 0.8) * (-1.0)
    loss = -min(unclipped, clipped)
    note = ""
    if r < 0.8:
        note = "<- clip! Don't let it shrink further"
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print("Interpretation:")
print("  * advantage>0 (good token): PPO encourages increasing ratio, but <=1.2")
print("  * advantage<0 (bad token): PPO encourages decreasing ratio, but >=0.8")
print("  * Clip acts as a safety net -> single-step updates won't be too aggressive, preventing training collapse")


#### 5.3 The Complete PPO Loss Formula

```
L_PPO = -E[ min(ratio x A, clip(ratio, 1-epsilon, 1+epsilon) x A) ]
        + beta x KL(pi_theta || pi_SFT)

Where:
  ratio = pi_theta(token|prompt) / pi_old(token|prompt)
  A = advantage = RM_score - baseline
  KL term = distributional difference between the new model and the SFT model
  beta = weight of the KL penalty (hyperparameter, typically 0.01-0.1)
```

The KL penalty is an easily overlooked but critically important part of PPO. Its role is not to improve performance, but to prevent the model from degrading.

The problem lies with the Reward Model. The RM is a scorer trained on human-annotated data -- it is only an approximate model of human preferences, not a perfect judge. If we optimize the LLM using only RM scores, the model will find loopholes in the RM's scoring mechanism and generate "score-gaming" responses that have high RM scores but terrible quality. This is called **Reward Hacking**.

For example, the RM might learn to give high scores to long responses (because annotators tend to choose more detailed answers). If optimized solely by RM scores, the model would learn to make responses endlessly long, even repeating content to pad the length -- the RM score would be high, but the response quality would be abysmal.

The KL penalty prevents this: it limits how far the new model can deviate from the SFT model. The SFT model may not distinguish good from bad, but at least it doesn't produce nonsense. The KL penalty essentially says: "You can optimize RM scores, but you can't deviate too far from the SFT model." The formula here is a teaching version; real PPO/RLHF typically also includes value functions, GAE, token-level advantages, clip/value losses, and other details. The larger beta is, the stronger the constraint, and the more conservative the model becomes.


## 6. DPO

#### 6.1 Problems with RLHF

RLHF is powerful, but engineering-wise it is very complex:
- Requires training a separate Reward Model (typically as many parameters as the main model)
- During PPO training, 4 models are live simultaneously: Actor (being trained), Reference (frozen SFT), Reward Model, Critic (value network)
- Many hyperparameters: KL coefficient, clip range, learning rate, rollout count...
- Unstable: PPO is notoriously "hard to tune"

In 2023, DPO (Direct Preference Optimization) proposed a simpler approach.

#### 6.2 The Mathematical Insight Behind DPO

The optimal policy of RLHF can be written as a function of the Reward Model and the reference policy. Through algebraic transformation, the Reward Model can be "eliminated" -- the result is the DPO loss:

```
L_DPO = -log( sigma(
    beta x log( pi_theta(chosen) / pi_ref(chosen) )
  - beta x log( pi_theta(rejected) / pi_ref(rejected) )
))
```

This formula looks intimidating, but the intuition is simple:

```
pi_theta(chosen) up   -> loss down  (good response, increase its probability!)
pi_theta(rejected) up -> loss up    (bad response, decrease its probability!)
pi_ref as denominator  -> prevents drifting too far (similar role to KL penalty)
beta -> controls "how aggressively" to change (larger beta = more aggressive)
```


In [ ]:
import math

# === DPO Loss Hand Calculation ===
print("=== DPO Loss Intuition Demo ===")
print()

def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5):
    """
    Simplified DPO loss
    logp = log(probability of generating this response)
    Larger values mean the model is more inclined to generate this response (less negative)
    """
    # Improvement of chosen relative to reference
    chosen_improvement = logp_chosen - logp_ref_chosen
    # Improvement of rejected relative to reference (we want this to decrease)
    rejected_improvement = logp_rejected - logp_ref_rejected
    
    # Core: chosen should improve more than rejected
    diff = beta * (chosen_improvement - rejected_improvement)
    
    # Sigmoid then take -log
    prob = 1 / (1 + math.exp(-diff))
    loss = -math.log(max(prob, 1e-10))
    
    return loss, chosen_improvement, rejected_improvement, diff, prob


scenarios = [
    ("Ideal: chosen up, rejected down",        -1.0, -5.0, -3.0, -3.0),
    ("OK: chosen up, rejected also up but less", -1.0, -2.5, -3.0, -3.0),
    ("Bad: chosen down, rejected up",           -5.0, -1.0, -3.0, -3.0),
    ("Both up, chosen rises more",              -0.5, -2.0, -3.0, -3.0),
    ("Both down, chosen falls less",            -4.5, -6.0, -3.0, -3.0),
]

print(f"{'Scenario':<38s} {'chosen_imp':>10s} {'rej_imp':>10s} {'diff':>10s} {'loss':>10s}")
print("-" * 74)

for desc, lc, lr, ref_c, ref_r in scenarios:
    loss, ci, ri, diff, prob = dpo_loss(lc, lr, ref_c, ref_r)
    print(f"{desc:<38s} {ci:>+10.2f} {ri:>+10.2f} {diff:>10.2f} {loss:>10.4f}")

print()
print("Core logic: DPO doesn't look at absolute values, only at chosen's improvement relative to rejected")
print("  More improvement for chosen + less improvement for rejected -> larger diff -> smaller loss")
print("  Even if both chosen and rejected probabilities are decreasing (model is forgetting),")
print("  as long as chosen decreases less than rejected -> diff > 0 -> loss still decreases")


## 7. RLHF vs DPO Comparison

| Dimension | RLHF (PPO) | DPO |
|:---|:---|:---|
| **Number of models** | 4 (Actor, Ref, RM, Critic) | 2 (Train, Ref) |
| **Training mode** | Online (each step requires model to generate new responses) | Offline (uses pre-collected preference pairs) |
| **Stability** | Poor (PPO is notoriously hard to tune) | Good (just supervised learning) |
| **Reward hacking** | High risk (RM is a fixed target) | Low risk (no RM to exploit) |
| **Theoretical ceiling** | Potentially higher (more online exploration) | Limited by preference data |
| **Debugging difficulty** | Hard (multiple components, hard to locate which one is broken) | Easy (one loss, debug like SFT) |
| **Who uses it** | OpenAI (GPT-4), Anthropic (Claude) | Open-source community (Zephyr, Qwen, etc.) |

**Recommendations**:
- Small teams, fast iteration -> DPO (less effort, comparable results)
- Large teams, pursuing maximum quality -> RLHF (large investment, higher ceiling)
- Middle ground -> Iterative DPO (multiple rounds of DPO, generating new preference pairs with the current model each round)


## 8. LLaMA 2's Complete Alignment Pipeline

```
Stage 1 -- Pretraining:
  LLaMA 2 Base, 2T tokens
  -> Can continue text, cannot converse

Stage 2 -- SFT:
  ~27K high-quality human-annotated conversations
  Trained for 2 epochs
  -> Can converse, follows instructions

Stage 3 -- Collect Preference Data:
  ~1M+ (prompt, chosen, rejected) pairs
  Annotators compare responses pairwise
  -> Human preference dataset

Stage 4a -- Reward Model:
  Initialized from SFT model
  Trained to score using preference data
  -> Automatic scorer

Stage 4b -- PPO:
  Score with RM -> PPO optimizes SFT model
  5 rounds of iteration
  -> LLaMA 2 Chat
```


## 9. Limitations and Applicable Scenarios of Alignment

RLHF/DPO makes models safer and more helpful, but also introduces new problems:

1. **Over-refusal**: The model may refuse harmless requests. For example, asking "how to perform CPR" might be misjudged as harmful medical advice and refused -- the model has over-associated "medical" with "harmful."
2. **Sycophancy**: If a user makes an error (e.g., "1+1=3"), the model doesn't correct them but instead agrees. This happens because annotators typically prefer responses that "don't contradict the user," and the RM has learned this preference.
3. **Style homogenization**: All responses become "polite safety-speak" -- starting with "Of course" or "I'd be happy to help" and ending with "I hope this helps." The diverse language style of the pretrained model is lost.
4. **Knowledge loss**: During alignment, the model may "forget" knowledge learned during pretraining. Because alignment data is typically concentrated on conversational and Q&A scenarios, the model's grasp of factual knowledge may degrade.

These problems are still actively researched, and there are no perfect solutions yet.

It's also worth emphasizing that alignment is not universally applicable. For tasks like code completion, translation, and text summarization, the definition of "good" is determined by objective downstream metrics (compilation pass rate, BLEU score, ROUGE score), and human preferences play a limited role. RLHF/DPO is primarily applicable to open-domain dialogue and instruction-following scenarios -- in these contexts, "good" is indeed subjective and needs human definition.


## Summary

Confirm that you understand these points (check in order):

1. ✅ Why alignment is needed: pretraining only teaches "how to speak," alignment teaches "how to speak appropriately"
2. ✅ 3H principles: Helpful (useful) + Honest (truthful) + Harmless (safe)
3. ✅ SFT: Uses high-quality conversations to teach dialogue format, but cannot distinguish good from bad
4. ✅ Preference data: (prompt, chosen, rejected) pairs, selected by annotators
5. ✅ Reward Model Loss: -log(σ(r_chosen - r_rejected)), turning preferences into classification
6. ✅ PPO Clip: ratio = π_new/π_old, constrained to [0.8, 1.2] to prevent overly aggressive updates
7. ✅ KL penalty: Prevents the model from gaming the score and drifting (reward hacking)
8. ✅ DPO: Skips RM+PPO, directly optimizes preference pairs -> simpler, comparable results
9. ✅ RLHF vs DPO: RLHF has a higher ceiling but is engineering-heavy; DPO offers better cost-effectiveness

**One-line summary**: Alignment = SFT foundation (learn to converse) -> RM establishes evaluation criteria -> PPO/DPO optimizes against those criteria. DPO makes this process accessible to small teams, not just large companies.


## Exercises

> You can ask AI to help explain the approach, but it's not recommended to let AI "complete the exercise for you."


**Exercise 1: Hand-calculate Reward Model Loss**

The Reward Model uses the Bradley-Terry model, with the loss formula: $$L = -\log(\sigma(r_{chosen} - r_{rejected}))$$ Given $r_{chosen} = 2.0$ and $r_{rejected} = -1.0$. Compute the loss by hand.

Hint: $\sigma(x) = 1 / (1 + e^{-x})$. First compute $r_{chosen} - r_{rejected} = 3.0$, then $\sigma(3.0)$, and finally $-\log(\sigma(3.0))$.


In [ ]:
# Exercise 1: Hand-calculate Reward Model Loss
import math

r_chosen = 2.0
r_rejected = -1.0

# TODO: Compute the reward difference between chosen and rejected
diff = None  # r_chosen - r_rejected
# TODO: Compute sigmoid(diff)
sigmoid_val = None  # 1 / (1 + exp(-diff))
# TODO: Compute loss = -log(sigmoid_val)
loss = None  # -log(sigmoid_val)

assert diff is not None, "Please compute diff first"
assert sigmoid_val is not None, "Please compute sigmoid first"
assert loss is not None, "Please compute loss first"

expected_diff = r_chosen - r_rejected
expected_sig = 1 / (1 + math.exp(-expected_diff))
expected_loss = -math.log(expected_sig)

assert diff == expected_diff, f"diff should be {expected_diff}"
assert abs(sigmoid_val - expected_sig) < 0.001, f"sigmoid should be {expected_sig:.4f}"
assert abs(loss - expected_loss) < 0.01, f"loss should be {expected_loss:.4f}"

print(f"✅ Exercise 1 passed:")
print(f"   diff = {diff:.1f}")
print(f"   sigmoid({diff:.1f}) = {sigmoid_val:.4f}")
print(f"   loss = -log({sigmoid_val:.4f}) = {loss:.4f}")
print("   When chosen is much better than rejected (large diff), loss is close to 0.")


**Exercise 2: PPO Clip Mechanism**

The core of PPO is the clip function, which limits the magnitude of policy updates: $\text{clip}(\text{ratio}, 1-\epsilon, 1+\epsilon)$. Given $\epsilon = 0.2$, compute the clipped result for the following three ratio values:

1. $\text{ratio} = 0.5$
2. $\text{ratio} = 1.1$
3. $\text{ratio} = 2.0$

Hint: clip simply constrains the value to the range $[1-\epsilon, 1+\epsilon] = [0.8, 1.2]$.


In [ ]:
# Exercise 2: PPO Clip Mechanism
epsilon = 0.2
ratio_1 = 0.5
ratio_2 = 1.1
ratio_3 = 2.0

# TODO: Clip the three ratio values to the range [0.8, 1.2]
clipped_1 = None  # Compute here
clipped_2 = None  # Compute here
clipped_3 = None  # Compute here

assert clipped_1 is not None, "Please compute clipped_1 first"
assert clipped_2 is not None, "Please compute clipped_2 first"
assert clipped_3 is not None, "Please compute clipped_3 first"

assert clipped_1 == 0.8, f"ratio=0.5 should clip to 0.8, you got {clipped_1}"
assert clipped_2 == 1.1, f"ratio=1.1 is within [0.8, 1.2], should not change"
assert clipped_3 == 1.2, f"ratio=2.0 should clip to 1.2, you got {clipped_3}"

print(f"✅ Exercise 2 passed:")
print(f"   ratio=0.5 -> clip=0.8 (update too small, pulled back to lower bound)")
print(f"   ratio=1.1 -> clip=1.1 (within safe range, unchanged)")
print(f"   ratio=2.0 -> clip=1.2 (update too large, pulled back to upper bound)")
print("   PPO clip prevents single updates from being too large, keeping training stable.")


**Exercise 3: DPO Loss Analysis**

The DPO loss formula is: $$L = -\log\frac{1}{1 + e^{-\beta(\Delta_{chosen} - \Delta_{rejected})}}$$ Given $\beta = 0.5$, $\Delta_{chosen} = 0.8$, and $\Delta_{rejected} = 0.2$. Compute the DPO loss.

Hint: First compute $\beta(\Delta_{chosen} - \Delta_{rejected}) = 0.5 \times 0.6 = 0.3$, then apply the sigmoid.


In [ ]:
# Exercise 3: DPO Loss Analysis
import math

beta = 0.5
delta_chosen = 0.8
delta_rejected = 0.2

# TODO: Compute beta * (delta_chosen - delta_rejected)
margin = None  # Compute here
# TODO: Compute loss = -log(sigmoid(margin))
sigmoid_m = None  # sigmoid(margin)
loss = None       # -log(sigmoid_m)

assert margin is not None, "Please compute margin first"
assert loss is not None, "Please compute loss first"

expected_margin = beta * (delta_chosen - delta_rejected)
expected_sig = 1 / (1 + math.exp(-expected_margin))
expected_loss = -math.log(expected_sig)

assert abs(margin - expected_margin) < 0.001, f"margin should be {expected_margin}"
assert abs(loss - expected_loss) < 0.01, f"loss should be {expected_loss:.4f}"

print(f"✅ Exercise 3 passed:")
print(f"   margin = β × (Δ_chosen - Δ_rejected) = {margin:.2f}")
print(f"   loss = -log(σ({margin:.2f})) = {loss:.4f}")
print("   DPO focuses on relative improvement: the more chosen improves over rejected, the smaller the loss.")
